# Ноутбук 02 — Разведочный анализ (EDA) с графиками

**EDA** (Exploratory Data Analysis) — это «осмотр данных глазами» до обучения. Мы ищем:
- как распределена цель (равномерно? со скошенными хвостами?);
- какие признаки сильнее связаны с целью (кандидаты в важные);
- как цель отличается по группам (богатые vs бедные и т.п.);
- согласованность данных (совпадают ли связанные колонки).

Словарик:
- **Гистограмма** — столбики: сколько значений попало в каждый интервал.
- **Корреляция Пирсона** — число от -1 до 1: насколько линейно связаны две величины.
- **Boxplot (ящик с усами)** — картинка распределения: линия внутри = медиана, ящик = средние 50% значений, усы = разброс.

In [ ]:
# Переходим в корень проекта (если надо) и импортируем библиотеки.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import pandas as pd  # таблицы
import numpy as np  # математика
import matplotlib.pyplot as plt  # графики (plt — стандартное короткое имя)
# Читаем данные.
train = pd.read_csv("hard_train.csv")
test = pd.read_csv("hard_test.csv")
# Цель — в отдельную переменную.
y = train["protection_score"]
print("train:", train.shape, " test:", test.shape)

## 1. Распределение цели: гистограмма

In [ ]:
# Создаём картинку размером 8x4 дюйма (figsize) — иначе будет мелко.
plt.figure(figsize=(8, 4))
# hist рисует гистограмму: bins=50 — разбить диапазон на 50 столбиков.
plt.hist(y, bins=50)
# Подписываем оси и заголовок — графики без подписей читать нельзя!
plt.xlabel("protection_score")  # подпись горизонтальной оси
plt.ylabel("Число клиентов")  # подпись вертикальной оси
plt.title("Распределение цели: сколько клиентов с каждой оценкой")  # заголовок
plt.show()  # показать картинку (в Jupyter можно и без этого, но так надёжнее)

Видно: большинство клиентов — в правой части (60–95), хвост тянется влево до ~3. То есть оценки смещены к высоким значениям. Это важно: модель, которая всем ставит ~70, уже не совсем плоха (но мы сделаем лучше!).

## 2. Какие признаки связаны с целью: топ корреляций

In [ ]:
# Берём только числовые колонки (текст коррелировать нельзя) и убираем саму цель.
numeric = train.select_dtypes("number").drop(columns=["protection_score"])
# corrwith(y) считает корреляцию Пирсона КАЖДОГО признака с целью.
# sort_values(key=np.abs, ascending=False) — сортируем по МОДУЛЮ (силе связи), сильные сверху.
# .head(15) — берём первые 15.
correlations = numeric.corrwith(y).sort_values(key=np.abs, ascending=False).head(15)
print(correlations.round(3))  # Печатаем числа, округлив до 3 знаков.

In [ ]:
# Рисуем те же корреляции столбиками: нагляднее, чем числа.
plt.figure(figsize=(8, 5))
# barh — ГОРИЗОНТАЛЬНЫЕ столбики; [::-1] переворачивает (лучшие окажутся сверху).
# color — красим: синим положительные связи, красным отрицательные.
colors = ["tab:red" if v < 0 else "tab:blue" for v in correlations[::-1]]
correlations[::-1].plot.barh(color=colors)
plt.xlabel("Корреляция Пирсона с protection_score")  # ось значений
plt.title("Топ-15 признаков по силе связи с целью")  # заголовок
plt.show()

Лидеры: `insurance_products` (сколько страховых продуктов, r ≈ 0.58), `active_policies`, `digital_behavior_score`, `cyber_protection`. Все положительные: больше продуктов/активности — выше оценка. Это логично и хорошо.

## 3. Связь глазами: scatter-графики

In [ ]:
# Scatter (точечная диаграмма): каждая точка — клиент (признак по горизонтали, цель по вертикали).
# Рисуем 2 графика рядом: subplots(1, 2) = 1 строка, 2 столбца; figsize — общий размер.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# График 1: цифровая активность vs цель. alpha=0.3 — прозрачность (точки наслаиваются).
axes[0].scatter(train["digital_behavior_score"], y, s=5, alpha=0.3)
axes[0].set_xlabel("digital_behavior_score")
axes[0].set_ylabel("protection_score")
axes[0].set_title("Чем активнее в цифре — тем выше оценка")
# График 2: использование приложения vs цель.
axes[1].scatter(train["mobile_app_usage"], y, s=5, alpha=0.3, color="tab:orange")
axes[1].set_xlabel("mobile_app_usage")
axes[1].set_ylabel("protection_score")
axes[1].set_title("Приложение тоже связано с оценкой")
plt.show()

Облака точек наклонены вправо-вверх — это и есть положительная связь. Но облака широкие: по одному признаку точно не предскажешь, нужны все вместе (этим и займётся модель).

## 4. Цель по группам: boxplot

In [ ]:
# Boxplot показывает распределение цели ВНУТРИ каждой группы.
# Как читать: линия в ящике — медиана; ящик — средние 50%; усы — разброс; точки — выбросы.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Слева: группы по числу страховых продуктов (0..8). Видно: чем больше продуктов, тем выше ящик.
train.boxplot(column="protection_score", by="insurance_products", ax=axes[0])
axes[0].set_title("Оценка по числу страховых продуктов")
axes[0].set_xlabel("insurance_products")
# Справа: группы по сегменту богатства.
train.boxplot(column="protection_score", by="wealth_segment", ax=axes[1])
axes[1].set_title("Оценка по сегменту богатства")
fig.suptitle("")  # Убираем лишний авто-заголовок pandas.
plt.show()

Группы заметно отличаются: у кого 8 продуктов — медиана около 90, у кого 0 — около 55. Значит, признаки несут сигнал. Но помни: boxplot показывает связь, а не причину (может, защищённые люди просто покупают больше).

## 5. Средние цели по текстовым группам (таблица)

In [ ]:
# groupby делит строки на группы по колонке; [цель].agg считает статистики в каждой группе.
# Пример: средняя оценка по занятости — business_owner vs unemployed отличаются сильно!
group_stats = train.groupby("employment")["protection_score"].agg(["mean", "std", "count"]).round(2)
# Сортируем по средней (по убыванию), чтобы лучшие группы были сверху.
print(group_stats.sort_values("mean", ascending=False))

## 6. Проверка согласованности данных

В хороших данных связанные колонки должны сходиться. Проверим три факта:
1. `insurance_products` = сумма 8 страховых флагов?
2. `average_claim_cost` пуст ровно когда `number_of_claims == 0`?
3. `claim_frequency` = claims / years (с округлением до 4 знаков)?

In [ ]:
# Список 8 страховых флагов 0/1.
flags = ["life_insurance", "property_insurance", "health_insurance", "travel_insurance",
        "car_insurance", "gadget_insurance", "cyber_protection", "identity_protection"]
# Сумма флагов по каждой строке (axis=1 = складывать вдоль строки).
flags_sum = train[flags].sum(axis=1)
# Сравниваем с insurance_products: (==) даёт True/False, .mean() — долю True. :.2% — формат процентов.
print(f"insurance_products == сумме флагов: {(flags_sum == train['insurance_products']).mean():.2%} строк")
# Проверка 2: пропуск там, где 0 обращений. Сравниваем два True/False массива поэлементно.
match = (train["average_claim_cost"].isna()) == (train["number_of_claims"] == 0)
print(f"average_claim_cost пуст ровно при 0 обращений: {match.mean():.2%} строк")
# Проверка 3: частота = обращения / годы. clip(lower=1) защищает от деления на 0.
# np.where(условие, если_да, если_нет): где стаж>0 — делим, иначе 0.
expected = np.where(train["years_with_company"] > 0,
                   train["number_of_claims"] / train["years_with_company"].clip(lower=1), 0.0)
# np.isclose — сравнение с допуском (данные округлены). atol=6e-5 — допуск.
print(f"claim_frequency == claims/years: {np.isclose(expected, train['claim_frequency'], atol=6e-5).mean():.2%} строк")

Все три проверки — 100%. Данные внутренне согласованы, формулы между колонками сходятся. Это хороший знак.

## 7. Полный EDA одним запуском: скрипт eda.py

Всё выше (и даже больше: анализ шума, выбросы) уже автоматизировано в `eda.py`. Запустим его и заглянем в отчёт — он сохранится в `/tmp/eda_demo/` (чтобы не трогать папку `reports/`).

In [ ]:
# Импортируем sys ради sys.executable — путь к текущему Python.
import sys
# Запускаем скрипт eda.py как из терминала: указываем train, test и папку для отчёта.
!{sys.executable} eda.py --train-csv hard_train.csv --test-csv hard_test.csv --output-dir /tmp/eda_demo
# Читаем начало получившегося отчёта и печатаем первые 40 строк.
report_lines = open("/tmp/eda_demo/eda_summary.md", encoding="utf-8").read().splitlines()
print("\n".join(report_lines[:40]))  # join склеивает строки через перевод строки.

## Выводы ноутбука 02

1. Цель смещена к высоким значениям (медиана ~76).
2. Сильнее связаны с целью: `insurance_products`, `active_policies`, `digital_behavior_score`, `cyber_protection`.
3. Группы отличаются (продукты, занятость, сегмент) — сигнал в данных есть.
4. Данные согласованы: формулы между колонками сходятся на 100%.
5. Полный отчёт + графики делает `eda.py` → `reports/eda/`.

Дальше — **03_feature_engineering**: создаём новые признаки и учимся обрабатывать пропуски.